In [1]:
from datasets import load_dataset, Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, RobertaTokenizer, pipeline, AutoModelForSeq2SeqLM
from peft import get_peft_model, LoraConfig, TaskType
from sklearn.metrics import accuracy_score
import torch
import evaluate
import random
import numpy as np
from tqdm import tqdm
import re

In [ ]:
def clean_text(text):
    text = text.lower()

    # 1. Remove URL / HTML
    text = re.sub(r"http\S+|www\S+", "", text)
    text = re.sub(r"<.*?>", "", text)

    # 2. Keep basic punctuation and remove strange symbols
    text = re.sub(r"[^a-z0-9.,'\s]", " ", text)

    # 3. Merge repeated words (more than 3 times)
    text = re.sub(r'\b(\w+)( \1\b){2,}', r'\1 \1', text)

    # 4. Delete consecutive stop word paragraphs (more than 5)
    stopwords = {"the", "of", "to", "in", "for", "on", "at", "from", "by", "as", "and", "with", "so", "but"}
    tokens = text.split()
    cleaned_tokens = []
    stop_run = 0
    for tok in tokens:
        if tok in stopwords:
            stop_run += 1
            if stop_run <= 3:
                cleaned_tokens.append(tok)
        else:
            stop_run = 0
            cleaned_tokens.append(tok)
    text = " ".join(cleaned_tokens)

    # 5. Delete particularly long words (suspected garbled characters)
    text = " ".join([t for t in text.split() if len(t) <= 20])

    # 6. Merge multiple spaces
    text = re.sub(r'\s+', ' ', text).strip()

    return text

# === 3. Defines a function to be used for map (applied to Dataset)===
def clean_example(example):
    example["text"] = clean_text(example["text"])
    return example

In [ ]:
dataset = load_dataset("ag_news")

# === 4. Filter both training and testing data ===
dataset = dataset.map(clean_example)
# ----------------------------------------------------------

# Load tokenizer and tokenize dataset
tokenizer = AutoTokenizer.from_pretrained("roberta-base")

def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True)

tokenized_datasets = dataset.map(tokenize_function, batched=True)

# Prepare data for PyTorch
tokenized_datasets = tokenized_datasets.remove_columns(["text"])
tokenized_datasets = tokenized_datasets.rename_column("label", "labels")
tokenized_datasets.set_format("torch")

train_dataset = tokenized_datasets["train"]
split_datasets = train_dataset.train_test_split(test_size=0.2, seed=42)
train_dataset = split_datasets['train']
eval_dataset = split_datasets['test']

test_dataset = tokenized_datasets["test"]

Map:   0%|          | 0/120000 [00:00<?, ? examples/s]

Map:   0%|          | 0/7600 [00:00<?, ? examples/s]

Map:   0%|          | 0/120000 [00:00<?, ? examples/s]

Map:   0%|          | 0/7600 [00:00<?, ? examples/s]

In [4]:
base_model = AutoModelForSequenceClassification.from_pretrained("roberta-base", num_labels=4)
for param in base_model.parameters():
    param.requires_grad = False

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [5]:
peft_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    inference_mode=False,
    r=8,                  # LoRA rank
    lora_alpha=16,        # Alpha scaling factor
    lora_dropout=0.1,     # Dropout for LoRA
    bias="none"
)

In [6]:
model = get_peft_model(base_model, peft_config)
model.print_trainable_parameters()

trainable params: 888,580 || all params: 125,537,288 || trainable%: 0.7078


In [7]:
metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = torch.argmax(torch.tensor(logits), dim=-1)
    return metric.compute(predictions=predictions, references=labels)

In [11]:
training_args = TrainingArguments(
    output_dir="./test_r",
    report_to=None,
    eval_strategy='steps',
    logging_steps=100,
    learning_rate=2e-5,
    weight_decay=0.01,
    num_train_epochs=3,
    max_steps=1200,
    use_cpu=False,
    dataloader_num_workers=4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    optim="adamw_torch",
    gradient_checkpointing=False,
    max_grad_norm=1.0,
    gradient_checkpointing_kwargs={'use_reentrant':True}
)


# Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset.shuffle(seed=42).select(range(20000)),  # Optional subset for speed
    eval_dataset=eval_dataset.select(range(2000)),                      # Optional subset for speed
    compute_metrics=compute_metrics,
)

trainer.train()

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
No label_names provided for model class `PeftModelForSequenceClassification`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism h

Epoch,Training Loss,Validation Loss,Accuracy
1,0.333300,0.344335,0.882500
2,0.317300,0.333197,0.890500
3,0.302100,0.331906,0.893500


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The 

TrainOutput(global_step=3750, training_loss=0.3927315132141113, metrics={'train_runtime': 1331.8329, 'train_samples_per_second': 45.051, 'train_steps_per_second': 2.816, 'total_flos': 1.595072987136e+16, 'train_loss': 0.3927315132141113, 'epoch': 3.0})

In [8]:
full_eval = trainer.evaluate(eval_dataset=test_dataset)

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The 

In [9]:
print(f"\n🧪 Final Test Accuracy: {full_eval['eval_accuracy'] * 100:.2f}%")


🧪 Final Test Accuracy: 87.88%


In [12]:
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer
import pandas as pd
import pickle

# === 1. Load test dataset from pickle ===
with open("test_unlabelled.pkl", "rb") as f:
    raw_dataset = pickle.load(f)

# This is a HuggingFace Dataset with field 'text'
test_texts = raw_dataset["text"]
#-------------------------------------------------------
test_texts = [clean_text(t) for t in test_texts]
#-------------------------------------------------------

# === 2. Tokenize and wrap as PyTorch dataset ===
tokenizer = AutoTokenizer.from_pretrained("roberta-base")

class TestDataset(Dataset):
    def __init__(self, texts, tokenizer, max_length=128):
        self.encodings = tokenizer(texts, truncation=True, padding="max_length", max_length=max_length, return_tensors="pt")

    def __len__(self):
        return len(self.encodings["input_ids"])

    def __getitem__(self, idx):
        return {k: v[idx] for k, v in self.encodings.items()}

test_dataset = TestDataset(test_texts, tokenizer)
test_loader = DataLoader(test_dataset, batch_size=64)

# === 3. Load your trained model ===
# Assume model is defined and loaded from previous training
model.eval()
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

# === 4. Predict labels ===
all_preds = []

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs["logits"] if isinstance(outputs, dict) else outputs.logits
        preds = torch.argmax(logits, dim=1)
        all_preds.extend(preds.cpu().numpy())

# === 5. Save to CSV ===
submission_df = pd.DataFrame({
    "ID": list(range(len(all_preds))),
    "Label": all_preds
})
submission_df.to_csv("submission.csv", index=False)

print("✅ submission.csv has been generated!")


✅ submission.csv has been generated!


In [14]:
test_texts[4]

"nba owners have imposed a luxury tax change on us based player draft stocks in talks to buy european buy up under national capital rule restriction changes yet now proposed by president on their future tax break reductions and may change further at beginning of august market start in the new era under change as recent small clubs market from low european league re structure draft investments and gain entry price support from higher paid investors where it cannot and high part is owned stock valued current state all players draft and rights while changes yet cannot hold huge multi national talent international contract in bigger nations by any rules or players choice of an increase while growing potential trade. increased potential in clubs' top income tier led recently loss cost cuting players overall increase no salary dropped players with small. nba has expressed opposition team increase even further need certain higher a big league. top player contract cost the new structure's mark